# 1.3 Web Search Integration

In this hands-on lab, you'll enhance an AI agent from a knowledge-limited chatbot to an intelligent assistant capable of accessing real-time web information. You'll see firsthand how tool integration transforms agent capabilities.

By the end of this lab, you will be able to:
- Build a basic LangChain agent using Azure OpenAI
- Understand the limitations of LLMs with static training data
- Integrate external web search tools to enable real-time information retrieval
- Compare agent behavior with and without tool augmentation
- Debug and trace agent reasoning using LangSmith

## 1.3.1 Without web search

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
import os
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model=os.getenv("LLM_MODEL"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL")
)

agent = create_agent(
    model=model
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)

In [ ]:
print(response['messages'][-1].content)

## 1.3.2 Add web search tool

In [ ]:
!pip install tavily

In [ ]:
from tavily import TavilyClient
client = TavilyClient()
response = client.search(
    query="What are the latest updates from NVIDIA ?"
)
print(response)

In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of San Francisco?")

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model=os.getenv("LLM_MODEL"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL")
)

agent = create_agent(
    model=model,
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)

In [ ]:
from pprint import pprint

pprint(response['messages'])

trace: https://smith.langchain.com/public/59432173-0dd6-49e8-9964-b16be6048426/r

In [ ]:
print(response['messages'][-1].content)